# Matched-pair recall-gap test using fine-grained features (CUB-200-2011)

We want to measure whether an attribute probe is truly using visual evidence for the attribute,
or whether it is partially relying on species identity as a shortcut.

Core idea:
For a given attribute and a pair of species (S1, S2), we build a matched test set where
attribute prevalence is identical in both species:
- same number of positive examples in S1 and S2
- same number of negative examples in S1 and S2

Then we evaluate:
- recall on positives for S1
- recall on positives for S2
- the recall gap |recall(S1) - recall(S2)|

If the recall gap is consistently large even after perfect prevalence matching,
that suggests the probe is using species-specific cues, not just attribute evidence.

We run this across:
- many attributes
- many species pairs
- multiple random seeds (because subsampling is random)
and summarize the recall gaps.


For each attribute:

1. Train a linear probe on top of frozen visual features.
2. Evaluate the probe on held-out test images.
3. Group test images by species.
4. For pairs of species:
   - Subsample images so that both species have the same number of
     attribute-positive and attribute-negative examples.
   - Compute recall on attribute-positive images for each species.
5. Measure the recall gap between species.

In [1]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn


In [2]:
ROOT = Path("/scratch/network/cr7998/cv_emergence_project")
CUB  = ROOT / "data" / "CUB_200_2011"

ATTR_TXT = ROOT / "data" / "attributes.txt"   # attr_id -> attr_name like has_primary_color::yellow

BASE_FEAT = ROOT / "features" / "resnet50_cub_fine"
CBM_FEAT  = ROOT / "features" / "resnet50_cub_cbm_fine"

assert CUB.exists(), f"Missing CUB folder: {CUB}"
assert ATTR_TXT.exists(), f"Missing attributes.txt: {ATTR_TXT}"
assert BASE_FEAT.exists(), f"Missing baseline fine features: {BASE_FEAT}"
assert CBM_FEAT.exists(), f"Missing cbm fine features: {CBM_FEAT}"

device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cpu'

In [3]:
def load_species_maps(cub_root: Path):
    """
    Loads species ID to name mappings from classes.txt.
    Also produces a prettified version for printing.
    """
    classes = pd.read_csv(
        cub_root / "classes.txt",
        sep=r"\s+",
        header=None,
        names=["species_id", "class_name"],
        engine="python"
    )

    def pretty(name: str) -> str:
        # Example: "001.Black_footed_Albatross" -> "Black footed albatross"
        return name.split(".", 1)[-1].replace("_", " ")

    id_to_pretty = {
        int(r.species_id): pretty(r.class_name)
        for _, r in classes.iterrows()
    }

    return id_to_pretty

species_id_to_name = load_species_maps(CUB)

def spname(sid: int) -> str:
    return species_id_to_name.get(int(sid), f"species_{sid}")


In [4]:
def load_meta(cub_root: Path) -> pd.DataFrame:
    """
    Returns a dataframe mapping each image to:
    - species ID
    - train/test split
    """
    img_species = pd.read_csv(
        cub_root / "image_class_labels.txt",
        sep=r"\s+",
        header=None,
        names=["image_id", "species_id"],
        engine="python"
    )

    split_df = pd.read_csv(
        cub_root / "train_test_split.txt",
        sep=r"\s+",
        header=None,
        names=["image_id", "is_train"],
        engine="python"
    )

    meta = img_species.merge(split_df, on="image_id")
    meta["species_name"] = meta["species_id"].map(spname)
    return meta

meta = load_meta(CUB)


In [5]:
def load_image_attr_labels_robust(cub_root: Path) -> pd.DataFrame:
    path = cub_root / "attributes" / "image_attribute_labels.txt"
    rows = []
    bad = 0

    with open(path, "r") as f:
        for line in f:
            toks = line.strip().split()
            if len(toks) < 4:
                bad += 1
                continue
            try:
                image_id = int(toks[0])
                attr_id  = int(toks[1])
                is_pres  = int(toks[2])
                cert     = int(toks[3])
                rows.append((image_id, attr_id, is_pres, cert))
            except:
                bad += 1

    df = pd.DataFrame(rows, columns=["image_id", "attr_id", "is_present", "certainty"])
    print("Parsed rows:", len(df), "bad lines skipped:", bad)
    return df

img_attr_long = load_image_attr_labels_robust(CUB)
img_attr_long.head()


Parsed rows: 3677856 bad lines skipped: 0


,image_id,attr_id,is_present,certainty
0,1,1,0,3
1,1,2,0,3
2,1,3,0,3
3,1,4,0,3
4,1,5,1,3


In [6]:
def load_attr_maps(attr_txt: Path):
    rows = []
    with open(attr_txt, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            aid_str, name = line.split(" ", 1)
            rows.append((int(aid_str), name))
    df = pd.DataFrame(rows, columns=["attr_id", "attr_name"])
    name_to_id = dict(zip(df["attr_name"], df["attr_id"]))
    id_to_name = dict(zip(df["attr_id"], df["attr_name"]))
    return df, name_to_id, id_to_name

attr_df, attr_name_to_id, attr_id_to_name = load_attr_maps(ATTR_TXT)
attr_df.head()


,attr_id,attr_name
0,1,has_bill_shape::curved_(up_or_down)
1,2,has_bill_shape::dagger
2,3,has_bill_shape::hooked
3,4,has_bill_shape::needle
4,5,has_bill_shape::hooked_seabird


In [7]:
ATTR_LIST = [
    "has_primary_color::yellow",
    "has_throat_color::yellow",
    "has_underparts_color::yellow",
    "has_belly_color::yellow",
    "has_breast_color::yellow",
]
for a in ATTR_LIST:
    assert a in attr_name_to_id, f"Missing attribute in attributes.txt: {a}"


In [8]:
# What this cell does:
# - For a given attribute_id, merges:
#   meta (image->species, split) with attribute labels (image->y)
# - Produces a clean table with y in {0,1}. where y is whether the attribut below is present or not.
# Why it matters:
# - This is the ground-truth label table used for training and evaluation.

def build_attr_labeled_df(meta: pd.DataFrame,
                          img_attr_long: pd.DataFrame,
                          attr_id: int,
                          min_certainty: int = 1) -> pd.DataFrame:
    """
    Returns dataframe with:
      image_id, species_id, species_name, is_train, y, certainty
    Only keeps annotations with certainty >= min_certainty.
    """
    sub = img_attr_long[img_attr_long["attr_id"] == int(attr_id)].copy()
    sub = sub[sub["certainty"] >= int(min_certainty)].copy()

    out = meta.merge(sub[["image_id", "is_present", "certainty"]], on="image_id", how="inner")
    out = out.rename(columns={"is_present": "y"})
    out["y"] = out["y"].astype(int)
    return out[["image_id", "species_id", "species_name", "is_train", "y", "certainty"]]

print("Defined:", "build_attr_labeled_df")

# sanity check on one attribute
attr_name = "has_primary_color::yellow"
aid = attr_name_to_id[attr_name]
lab = build_attr_labeled_df(meta, img_attr_long, aid, min_certainty=1)
print("Attribute:", attr_name, "rows labeled:", len(lab), "pos rate:", lab.y.mean())
lab.head()


Defined: build_attr_labeled_df
Attribute: has_primary_color::yellow rows labeled: 11788 pos rate: 0.1506616898540889


,image_id,species_id,species_name,is_train,y,certainty
0,1,1,Black footed Albatross,0,0,3
1,2,1,Black footed Albatross,1,0,4
2,3,1,Black footed Albatross,0,0,4
3,4,1,Black footed Albatross,1,0,4
4,5,1,Black footed Albatross,1,0,4


In [9]:
# - Loads a feature tensor from disk and converts it to float32 torch.Tensor.

def safe_torch_load(path: Path):
    """
    Uses weights_only=True if supported to reduce pickle risk warnings.
    """
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        return torch.load(path, map_location="cpu")

def load_features(feat_dir: Path, layer: str, split: str) -> torch.Tensor:
    """
    Loads feature tensor saved as {layer}_{split}.pt from feat_dir.
    """
    p = feat_dir / f"{layer}_{split}.pt"
    assert p.exists(), f"Missing: {p}"
    X = safe_torch_load(p)
    if not isinstance(X, torch.Tensor):
        X = torch.tensor(X)
    return X.float()

In [10]:
import numpy as np
import torch

def to_1d_int_array(x):
    """Convert tensor/list/np array to 1D int numpy array."""
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    x = np.array(x)
    x = x.reshape(-1)
    return x.astype(int)

def load_split_order(feat_dir, split):
    p = feat_dir / f"labels_{split}.pt"
    assert p.exists(), f"Missing: {p}"
    t = torch.load(p, map_location="cpu", weights_only=True)

    assert isinstance(t, dict), f"Expected dict in {p}, got {type(t)}"
    assert "image_ids" in t, f"{p} missing 'image_ids' key; has {list(t.keys())}"

    ids = to_1d_int_array(t["image_ids"])
    kind = infer_kind(ids)
    return kind, ids


def infer_kind(arr):
    # Heuristic:
    # - species ids: 1..200 (sometimes 0..199)
    # - image ids: 1..11788
    if arr.max() <= 200 and arr.min() >= 0:
        return "species_id_like"
    if arr.max() > 200:
        return "image_id_like"
    return "unknown"


In [11]:
LAYER = "layer4.0"

base_kind_tr, base_ids_tr = load_split_order(BASE_FEAT, "train")
base_kind_te, base_ids_te = load_split_order(BASE_FEAT, "test")
cbm_kind_tr,  cbm_ids_tr  = load_split_order(CBM_FEAT, "train")
cbm_kind_te,  cbm_ids_te  = load_split_order(CBM_FEAT, "test")

print(
    "Baseline train:",
    base_kind_tr,
    "id range:",
    (base_ids_tr.min(), base_ids_tr.max())
)

print(
    "Baseline test:",
    base_kind_te,
    "id range:",
    (base_ids_te.min(), base_ids_te.max())
)

Baseline train: image_id_like id range: (2, 11787)
Baseline test: image_id_like id range: (1, 11788)


In [12]:
# What this cell does:
# - Aligns features (in feature row order) to attribute labels by image_id.
# - Produces X_aligned and df_aligned with the same ordering.


def align_features_and_labels(X_split: torch.Tensor,
                              image_ids_in_feature_order: np.ndarray,
                              labeled_df_split: pd.DataFrame):
    """
    Inputs:
      X_split: feature tensor of shape [N, D]
      image_ids_in_feature_order: length N, image_id for each row of X_split
      labeled_df_split: dataframe with at least columns [image_id, y, species_id, species_name]

    Output:
      X_aligned: features for images that have labels
      df_aligned: same rows, same order, includes y and species info
    """
    labeled = labeled_df_split.set_index("image_id")[["y", "species_id", "species_name"]]

    keep_idx = []
    rows = []
    for i, img_id in enumerate(image_ids_in_feature_order):
        img_id = int(img_id)
        if img_id in labeled.index:
            keep_idx.append(i)
            y, sid, sname = labeled.loc[img_id]
            rows.append((img_id, int(sid), str(sname), int(y)))

    X_aligned = X_split[keep_idx]
    df_aligned = pd.DataFrame(rows, columns=["image_id", "species_id", "species_name", "y"])
    return X_aligned, df_aligned

In [13]:
# What this cell does:
# - Defines a linear probe (single linear layer).
# - Trains it using BCEWithLogitsLoss with mild class-imbalance handling.
# Why it matters:
# - Probe is the measurement instrument for "is the attribute encoded in features?"

class LinearProbe(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.lin = nn.Linear(d, 1)

    def forward(self, x):
        return self.lin(x).squeeze(-1)

def train_probe(Xtr: torch.Tensor, ytr: np.ndarray,
                seed=0, lr=1e-2, wd=1e-4, epochs=25, batch=512):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    Xtr = Xtr.to(device)
    ytr_t = torch.tensor(ytr, dtype=torch.float32, device=device)

    probe = LinearProbe(Xtr.shape[1]).to(device)
    opt = torch.optim.AdamW(probe.parameters(), lr=lr, weight_decay=wd)

    pos = float(ytr_t.mean().item())
    pos_weight = torch.tensor([(1 - pos) / pos], device=device) if 0 < pos < 1 else torch.tensor([1.0], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    n = Xtr.shape[0]
    for _ in range(epochs):
        perm = torch.randperm(n, device=device)
        for i in range(0, n, batch):
            idx = perm[i:i+batch]
            logits = probe(Xtr[idx])
            loss = loss_fn(logits, ytr_t[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()

    return probe

@torch.no_grad()
def predict_probs(probe: nn.Module, X: torch.Tensor, batch=4096) -> np.ndarray:
    probe.eval()
    probs = []
    for i in range(0, X.shape[0], batch):
        xb = X[i:i+batch].to(device)
        logits = probe(xb)
        probs.append(torch.sigmoid(logits).detach().cpu())
    return torch.cat(probs, dim=0).numpy()

print("Defined:", "LinearProbe", "train_probe", "predict_probs")


Defined: LinearProbe train_probe predict_probs


1. Identify species that have both positive and negative examples.
2. For each pair:
   - Subsample so both species have identical numbers of positives and negatives.
3. Compute recall on positive examples for each species.
4. Measure the absolute recall difference.

In [14]:
# What this cell does:
# - Finds species that have enough positives and negatives for the chosen attribute.
# - Samples many species pairs.
# - For each pair, subsamples to match prevalence exactly and computes recall on positives.
# Why it matters:
# - This isolates species-specific differences even when prevalence is controlled perfectly.

def make_candidate_pairs(df_test: pd.DataFrame, min_each=10, max_pairs=200, seed=0):
    """
    Returns list of tuples (sid_A, sid_B, mpos, mneg) where:
      mpos = min(posA, posB)
      mneg = min(negA, negB)
    and both are >= min_each.
    """
    g = df_test.groupby("species_id")["y"].agg(["count", "sum"]).rename(columns={"sum": "pos"})
    g["neg"] = g["count"] - g["pos"]
    ok = g[(g["pos"] >= min_each) & (g["neg"] >= min_each)]
    sids = ok.index.to_list()

    rng = np.random.default_rng(seed)
    pairs = []
    if len(sids) < 2:
        return pairs

    for _ in range(max_pairs * 10):
        a, b = rng.choice(sids, size=2, replace=False)
        mpos = int(min(ok.loc[a, "pos"], ok.loc[b, "pos"]))
        mneg = int(min(ok.loc[a, "neg"], ok.loc[b, "neg"]))
        if mpos >= min_each and mneg >= min_each:
            pairs.append((int(a), int(b), mpos, mneg))
        if len(pairs) >= max_pairs:
            break
    return pairs

def matched_pair_eval(df_test: pd.DataFrame, probs: np.ndarray, sid_A: int, sid_B: int,
                      mpos: int, mneg: int, seed=0, thr=0.5):
    """
    Subsamples to:
      mpos positives + mneg negatives from each species.
    Computes recall on positive examples only for each species subset.
    """
    df = df_test.copy()
    df["prob"] = probs

    A = df[df.species_id == sid_A]
    B = df[df.species_id == sid_B]

    A_pos, A_neg = A[A.y == 1], A[A.y == 0]
    B_pos, B_neg = B[B.y == 1], B[B.y == 0]

    A_s = pd.concat([A_pos.sample(mpos, random_state=seed), A_neg.sample(mneg, random_state=seed)])
    B_s = pd.concat([B_pos.sample(mpos, random_state=seed), B_neg.sample(mneg, random_state=seed)])

    def recall_pos(d):
        pos = d[d.y == 1]
        pred = (pos.prob.values >= thr).astype(int)
        return float((pred == 1).mean()) if len(pos) else np.nan

    recA = recall_pos(A_s)
    recB = recall_pos(B_s)

    return {
        "sid_A": sid_A,
        "sid_B": sid_B,
        "species_A": spname(sid_A),
        "species_B": spname(sid_B),
        "npos": int(mpos),
        "nneg": int(mneg),
        "recall_A": float(recA),
        "recall_B": float(recB),
        "gap": float(abs(recA - recB)),
    }

def matched_pair_bootstrap_summary(
    df_te: pd.DataFrame,
    probs: np.ndarray,
    pairs,
    *,
    thr=0.5,
    B=300,
):
    """
    For each candidate (sid_A, sid_B, mpos, mneg):
      - run matched_pair_eval B times (each time resampling matched subsets)
      - summarize distribution of 'gap' across bootstrap runs

    Returns:
      res_long: one row per (pair, bootstrap_run) with recall_A, recall_B, gap
      pair_summary: one row per pair with gap_mean/std/CI/p and stability metrics
    """
    matched_cols = [
        "sid_A", "sid_B", "species_A", "species_B",
        "npos", "nneg", "recall_A", "recall_B", "gap", "boot_id"
    ]
    summary_cols = [
        "sid_A","sid_B","species_A","species_B",
        "npos","nneg",
        "gap_mean","gap_std","gap_ci_lo","gap_ci_hi","gap_p",
        "gap_ci_width","gap_snr","gap_norm","n_runs"
    ]

    if pairs is None or len(pairs) == 0:
        return pd.DataFrame(columns=matched_cols), pd.DataFrame(columns=summary_cols)

    rows = []
    for (a, b, mpos, mneg) in pairs:
        for boot_id in range(B):
            r = matched_pair_eval(
                df_te, probs,
                sid_A=int(a), sid_B=int(b),
                mpos=int(mpos), mneg=int(mneg),
                seed=int(boot_id), thr=float(thr)
            )
            r["boot_id"] = int(boot_id)
            rows.append(r)

    res_long = pd.DataFrame(rows)

    if res_long.empty:
        return res_long, pd.DataFrame(columns=summary_cols)

    def _ci_lo(x): return bootstrap_ci(x)[0]
    def _ci_hi(x): return bootstrap_ci(x)[1]

    pair_summary = (
        res_long.groupby(["sid_A","sid_B","species_A","species_B"], as_index=False)
                .agg(
                    npos=("npos","min"),
                    nneg=("nneg","min"),
                    gap_mean=("gap","mean"),
                    gap_std=("gap","std"),
                    gap_ci_lo=("gap", _ci_lo),
                    gap_ci_hi=("gap", _ci_hi),
                    gap_p=("gap", bootstrap_p_value),
                    n_runs=("gap","size"),
                )
    )

    EPS = 1e-12
    pair_summary["gap_ci_width"] = pair_summary["gap_ci_hi"] - pair_summary["gap_ci_lo"]

    # Stability metric: big gap relative to variability
    pair_summary["gap_snr"] = pair_summary["gap_mean"] / (pair_summary["gap_std"].fillna(0.0) + EPS)

    # Normalized effect size on 0..1 gap scale (gap itself already in [0,1])
    pair_summary["gap_norm"] = pair_summary["gap_mean"]

    pair_summary = pair_summary.sort_values("gap_mean", ascending=False).reset_index(drop=True)

    return res_long, pair_summary


def species_recall_prevalence_table(df_te: pd.DataFrame, probs: np.ndarray, thr=0.5) -> pd.DataFrame:
    """
    Per-species table on the TEST set:
      - n, n_pos, n_neg
      - prevalence = n_pos / n
      - tp = # of positives predicted positive
      - recall = tp / n_pos
      - precision = tp / n_pred_pos   (optional but cheap and often useful)
    """
    df = df_te[["species_id", "species_name", "y"]].copy()
    df["prob"] = np.asarray(probs, dtype=float)
    df["pred"] = (df["prob"] >= thr).astype(int)

    # Basic counts per species
    g = (df.groupby(["species_id", "species_name"], as_index=False)
           .agg(
               n=("y", "size"),
               n_pos=("y", "sum"),
               n_pred_pos=("pred", "sum"),
           ))
    g["n_neg"] = g["n"] - g["n_pos"]
    g["prevalence"] = g["n_pos"] / g["n"]

    # True positives per species (only among y==1)
    tp = (df[df["y"] == 1]
            .groupby(["species_id", "species_name"])["pred"]
            .sum()
            .reset_index(name="tp"))

    out = g.merge(tp, on=["species_id", "species_name"], how="left")
    out["tp"] = out["tp"].fillna(0).astype(int)

    # Recall: tp / n_pos (handle n_pos==0)
    out["recall"] = np.where(out["n_pos"] > 0, out["tp"] / out["n_pos"], np.nan)

    # Precision: tp / n_pred_pos (handle n_pred_pos==0)
    out["precision"] = np.where(out["n_pred_pos"] > 0, out["tp"] / out["n_pred_pos"], np.nan)

    out = out.sort_values(["n"], ascending=False).reset_index(drop=True)
    return out


def add_species_bootstrap_ci(df_te: pd.DataFrame, probs: np.ndarray, thr=0.5, B=300, min_pos_for_ci=1) -> pd.DataFrame:
    """
    Adds bootstrap CIs for per-species recall.
    Bootstraps *within each species* by resampling that species' test images with replacement.

    Outputs new columns:
      - recall_ci_lo, recall_ci_hi
      - recall_bs_mean (bootstrap mean; usually close to recall)
      - recall_ci_width
    """
    df = df_te[["species_id", "species_name", "y"]].copy()
    df["prob"] = np.asarray(probs, dtype=float)

    rows = []
    rng = np.random.default_rng(0)

    for (sid, sname), d in df.groupby(["species_id", "species_name"]):
        d = d.reset_index(drop=True)
        n = len(d)
        n_pos = int(d["y"].sum())

        # If no positives, recall undefined (and CI meaningless)
        if n_pos < min_pos_for_ci:
            rows.append({
                "species_id": int(sid),
                "species_name": str(sname),
                "recall_bs_mean": np.nan,
                "recall_ci_lo": np.nan,
                "recall_ci_hi": np.nan,
                "recall_ci_width": np.nan,
                "B": int(B),
            })
            continue

        vals = []
        for b in range(B):
            idx = rng.integers(0, n, size=n)  # resample rows with replacement
            s = d.iloc[idx]

            pos = s[s["y"] == 1]
            if len(pos) == 0:
                vals.append(np.nan)
                continue
            pred_pos = (pos["prob"].to_numpy() >= thr).astype(int)
            vals.append(float(pred_pos.mean()))

        vals = np.asarray(vals, dtype=float)
        lo, hi = bootstrap_ci(vals, alpha=0.05)
        rows.append({
            "species_id": int(sid),
            "species_name": str(sname),
            "recall_bs_mean": float(np.nanmean(vals)),
            "recall_ci_lo": lo,
            "recall_ci_hi": hi,
            "recall_ci_width": (hi - lo) if (np.isfinite(lo) and np.isfinite(hi)) else np.nan,
            "B": int(B),
        })

    ci_df = pd.DataFrame(rows)
    base = species_recall_prevalence_table(df_te, probs, thr=thr)
    out = base.merge(ci_df, on=["species_id", "species_name"], how="left")
    return out


In [15]:
import numpy as np

def bootstrap_ci(x, alpha=0.05):
    x = np.asarray(x, dtype=float)
    return (float(np.quantile(x, alpha/2)),
            float(np.quantile(x, 1 - alpha/2)))

def bootstrap_p_value(x):
    x = np.asarray(x, dtype=float)
    p_lo = float(np.mean(x <= 0))
    p_hi = float(np.mean(x >= 0))
    return 2.0 * min(p_lo, p_hi)


In [16]:
def bootstrap_ci(x, alpha=0.05):
    """
    Percentile bootstrap CI for a 1D array x.
    Returns (lo, hi). If x is empty or all-nan, returns (nan, nan).
    """
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if x.size == 0:
        return (np.nan, np.nan)
    lo = np.quantile(x, alpha/2)
    hi = np.quantile(x, 1 - alpha/2)
    return (float(lo), float(hi))

def bootstrap_p_value(values, null=0.0):
    """
    Two-sided bootstrap p-value for H0: E[value] == null
    Using bootstrap distribution of the statistic itself.

    p = 2 * min(P(value <= null), P(value >= null))
    """
    v = np.asarray(values, dtype=float)
    v = v[~np.isnan(v)]
    if v.size == 0:
        return np.nan
    p_lo = np.mean(v <= null)
    p_hi = np.mean(v >= null)
    return float(2.0 * min(p_lo, p_hi))

def safe_div(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    out = np.full_like(a, np.nan, dtype=float)
    m = b != 0
    out[m] = a[m] / b[m]
    return out


In [17]:
def run_one_attribute(
    attr_name: str,
    feat_dir: Path,
    split_order_kind_train: str,
    split_order_train: np.ndarray,
    split_order_kind_test: str,
    split_order_test: np.ndarray,
    layer: str,
    *,
    min_certainty: int = 1,
    thr: float = 0.5,
    epochs: int = 25,
    min_each: int = 10,
    n_pairs: int = 200,
    B_gap: int = 100,
    B_species: int = 100,
):
    """
    Runs the full pipeline for ONE attribute and ONE model's features:
      1) Build labeled train/test sets for this attribute (certainty filtering)
      2) Load train/test features for the chosen layer
      3) Align features to labels by image_id using split order arrays
      4) Train a linear probe on train features
      5) Predict probabilities on test features
      6) Build per-species prevalence+recall table (and bootstrap CI for recall)
      7) (Optional) Generate candidate species pairs for matched evaluation
      8) (Optional) Bootstrap matched-pair recall gap per pair (CI + p-value + stability metrics)
    """

    # Alignment requires feature order indexed by image_id.
    assert split_order_kind_train == "image_id_like" and split_order_kind_test == "image_id_like", (
        "Cannot align features to attribute labels because labels_{split}.pt is not image_id-like.\n"
        "If you hit this, we need to read the dataset ordering from your extractor code."
    )

    # Build per-image labels for this attribute.
    aid = attr_name_to_id[attr_name]
    lab = build_attr_labeled_df(meta, img_attr_long, aid, min_certainty=min_certainty)
    lab_train = lab[lab["is_train"] == 1].copy()
    lab_test  = lab[lab["is_train"] == 0].copy()

    # Load precomputed features.
    Xtr_all = load_features(feat_dir, layer, "train")
    Xte_all = load_features(feat_dir, layer, "test")

    # Align labeled images to feature tensor order.
    Xtr, df_tr = align_features_and_labels(Xtr_all, split_order_train, lab_train)
    Xte, df_te = align_features_and_labels(Xte_all, split_order_test,  lab_test)

    ytr = df_tr["y"].astype(int).to_numpy()
    yte = df_te["y"].astype(int).to_numpy()

    # Train probe + predict probabilities on test.
    probe = train_probe(Xtr, ytr, seed=0, epochs=epochs)
    probs = predict_probs(probe, Xte)

    # Species table (point estimates + bootstrap CI).
    species_table = add_species_bootstrap_ci(df_te, probs, thr=thr, B=B_species)

    # Overall accuracy at threshold (headline only; prof cares more about per-species table).
    test_acc = float(((probs >= thr).astype(int) == yte).mean()) if len(yte) else np.nan

    # Matched pairs (optional).
    if n_pairs is None or int(n_pairs) <= 0:
        res_long = pd.DataFrame()
        pair_summary = pd.DataFrame()
        pairs = []
    else:
        pairs = make_candidate_pairs(df_te, min_each=min_each, max_pairs=n_pairs, seed=0)
        res_long, pair_summary = matched_pair_bootstrap_summary(
            df_te, probs, pairs, thr=thr, B=B_gap
        )

    mean_gap = float(pair_summary["gap_mean"].mean()) if (pair_summary is not None and len(pair_summary)) else np.nan
    p90_gap  = float(pair_summary["gap_mean"].quantile(0.9)) if (pair_summary is not None and len(pair_summary)) else np.nan

    info = {
        "attr": attr_name,
        "layer": layer,
        "n_train": int(len(df_tr)),
        "n_test": int(len(df_te)),
        "train_pos_rate": float(ytr.mean()) if len(ytr) else np.nan,
        "test_pos_rate": float(yte.mean()) if len(yte) else np.nan,
        "test_acc": float(test_acc),
        "thr": float(thr),
        "epochs": int(epochs),
        "n_pairs": int(len(pairs)),
        "B_gap": int(B_gap),
        "B_species": int(B_species),
        "mean_gap": mean_gap,
        "p90_gap": p90_gap,
    }

    return info, res_long, pair_summary, df_te, species_table


In [18]:
def screen_attributes_for_species_variation(
    candidate_attrs,
    feat_dir: Path,
    kind_tr: str, ids_tr: np.ndarray,
    kind_te: str, ids_te: np.ndarray,
    layer: str,
    *,
    min_certainty: int = 1,
    thr: float = 0.5,
    min_pos_per_species: int = 10,
    min_species_with_pos: int = 15,
    min_overall_prev: float = 0.05,
    max_overall_prev: float = 0.95,
    epochs: int = 8,
    max_attrs: int | None = None,
    verbose_every: int = 50,
    keep_error_examples: int = 5,
    B_species: int = 200,   # NEW: bootstrap trials for species recall CI during screening
):
    rows = []
    errors = []
    stats = {
        "tried": 0,
        "success": 0,
        "filtered_too_few_species_pos": 0,
        "filtered_prev_out_of_range": 0,
        "filtered_no_recall_vals": 0,
        "errored": 0,
    }

    cand = list(candidate_attrs)
    if max_attrs is not None:
        cand = cand[:max_attrs]

    for i, attr in enumerate(cand):
        stats["tried"] += 1
        try:
            info, _, _, _, species_table = run_one_attribute(
                attr,
                feat_dir,
                kind_tr, ids_tr,
                kind_te, ids_te,
                layer=layer,
                min_certainty=min_certainty,
                thr=thr,
                epochs=epochs,
                min_each=10,
                n_pairs=0,          # screening: skip matched pairs
                B_species=B_species,
            )

            st = species_table.copy()
            overall_prev = float(st["n_pos"].sum() / st["n"].sum()) if st["n"].sum() > 0 else np.nan

            st_pos = st[st["n_pos"] >= min_pos_per_species].copy()
            n_species_pos = int(len(st_pos))

            if n_species_pos < min_species_with_pos:
                stats["filtered_too_few_species_pos"] += 1
                continue

            if not (min_overall_prev <= overall_prev <= max_overall_prev):
                stats["filtered_prev_out_of_range"] += 1
                continue

            recall_vals = st_pos["recall"].dropna().to_numpy()
            if recall_vals.size == 0:
                stats["filtered_no_recall_vals"] += 1
                continue

            stats["success"] += 1

            recall_std = float(np.std(recall_vals))
            recall_range = float(np.max(recall_vals) - np.min(recall_vals))
            recall_p90_p10 = float(np.quantile(recall_vals, 0.9) - np.quantile(recall_vals, 0.1))

            rows.append({
                "attr": attr,
                "overall_prev": overall_prev,
                "n_species_pos": n_species_pos,
                "recall_std": recall_std,
                "recall_range": recall_range,
                "recall_p90_p10": recall_p90_p10,
                "test_acc": float(info["test_acc"]),
                "n_test": int(info["n_test"]),
            })

            if verbose_every and ((i + 1) % verbose_every == 0):
                print(f"[{i+1}/{len(cand)}] ok: {attr}  prev={overall_prev:.3f}  n_species_pos={n_species_pos}")

        except Exception as e:
            stats["errored"] += 1
            if len(errors) < keep_error_examples:
                errors.append((attr, repr(e)))
            continue

    screen_df = pd.DataFrame(rows)

    print("\n--- Screening summary ---")
    for k, v in stats.items():
        print(f"{k}: {v}")
    if errors:
        print("\nExample errors (first few):")
        for a, msg in errors:
            print(" ", a, "->", msg)

    if screen_df.empty:
        print("\nNo attributes passed filters. Likely causes:")
        print(" - run_one_attribute is erroring for most attrs (see errors above)")
        print(" - filters too strict for your attribute distribution")
        return screen_df

    screen_df = screen_df.sort_values(
        ["recall_p90_p10", "recall_range", "recall_std"],
        ascending=False
    ).reset_index(drop=True)

    return screen_df


# ---- Run it ----
CANDIDATE_ATTRS = attr_df["attr_name"].tolist()

screen_df = screen_attributes_for_species_variation(
    CANDIDATE_ATTRS,
    feat_dir=BASE_FEAT,
    kind_tr=base_kind_tr, ids_tr=base_ids_tr,
    kind_te=base_kind_te, ids_te=base_ids_te,
    layer=LAYER,
    min_certainty=1,
    thr=0.5,
    min_pos_per_species=10,
    min_species_with_pos=15,
    min_overall_prev=0.05,
    max_overall_prev=0.95,
    epochs=8,
    max_attrs=200,     
    verbose_every=25,
)

if screen_df.empty:
    # Loosen constraints automatically so you get *something*
    screen_df = screen_attributes_for_species_variation(
        CANDIDATE_ATTRS,
        feat_dir=BASE_FEAT,
        kind_tr=base_kind_tr, ids_tr=base_ids_tr,
        kind_te=base_kind_te, ids_te=base_ids_te,
        layer=LAYER,
        min_certainty=1,
        thr=0.5,
        min_pos_per_species=5,
        min_species_with_pos=8,
        min_overall_prev=0.02,
        max_overall_prev=0.98,
        epochs=6,
        max_attrs=200,
        verbose_every=25,
    )

if not screen_df.empty:
    TOP_K = 12
    ATTR_LIST = screen_df["attr"].head(TOP_K).tolist()
    print("\nNew ATTR_LIST:")
    for a in ATTR_LIST:
        print(" ", a)
    screen_df.head(20)


# Map: attr -> typical cross-species recall spread (from screening)
# This is the scale we normalize gaps against.
attr_to_spread = screen_df.set_index("attr")["recall_p90_p10"].to_dict()


[150/200] ok: has_bill_length::about_the_same_as_head  prev=0.366  n_species_pos=91

--- Screening summary ---
tried: 200
success: 73
filtered_too_few_species_pos: 127
filtered_prev_out_of_range: 0
filtered_no_recall_vals: 0
errored: 0

New ATTR_LIST:
  has_upper_tail_color::white
  has_bill_shape::all-purpose
  has_breast_pattern::solid
  has_upperparts_color::black
  has_bill_length::shorter_than_head
  has_upperparts_color::brown
  has_wing_color::black
  has_breast_color::white
  has_wing_color::grey
  has_wing_color::white
  has_under_tail_color::black
  has_under_tail_color::white


In [19]:
# Runs the matched-pair pipeline across multiple attributes (ATTR_LIST)
# for both models:
# - baseline features (BASE_FEAT)
# - CBM features (CBM_FEAT)
#
# run_many(...) loops attributes, calls run_one_attribute(...),
# stores:
# - info_df: per-attribute run metadata (acc, mean gap, etc.)
# - pairs_df: per-(attr, pair) gap_mean/gap_std results
#
# This produces baseline_info/baseline_pairs and cbm_info/cbm_pairs
# for downstream comparison and reporting.

#   Meaning of printed numbers:
#     - test_acc: test-set accuracy of the attribute probe/classifier for this attribute
#                 (on this model's features at the chosen layer)
#     - mean_gap: average matched-pair recall gap across the sampled species pairs
#                 for this attribute (higher = more species-dependent / entangled)

# Choose a fine layer to use consistently
LAYER = "layer4.0"

# Baseline split order (must be image ids)
base_kind_tr, base_ids_tr = load_split_order(BASE_FEAT, "train")
base_kind_te, base_ids_te = load_split_order(BASE_FEAT, "test")

# CBM split order (must be image ids)
cbm_kind_tr, cbm_ids_tr = load_split_order(CBM_FEAT, "train")
cbm_kind_te, cbm_ids_te = load_split_order(CBM_FEAT, "test")

def run_many(
    attr_list,
    model_name,
    feat_dir,
    kind_tr, ids_tr,
    kind_te, ids_te,
    *,
    layer,
    min_certainty=1,
    thr=0.5,
    epochs=25,
    min_each=10,
    n_pairs=200,
    B_gap=100,
    B_species=100,
):
    """
    Runs run_one_attribute over a list of attrs.
    Collects:
      - info_df: one row per attr (headline metrics)
      - pairs_df: per-(attr, pair) summary table (gap_mean, CI, p, etc.)
      - species_df: per-(attr, species) table (prevalence, recall, recall CI)
    """
    all_info = []
    all_pair_summ = []
    all_species = []

    for attr in attr_list:
        info, _, pair_summ, _, species_table = run_one_attribute(
            attr, feat_dir,
            kind_tr, ids_tr,
            kind_te, ids_te,
            layer=layer,
            min_certainty=min_certainty,
            thr=thr,
            epochs=epochs,
            min_each=min_each,
            n_pairs=n_pairs,
            B_gap=B_gap,
            B_species=B_species,
        )

        info = dict(info)
        info["model"] = model_name
        all_info.append(info)

        if pair_summ is not None and len(pair_summ):
            ps = pair_summ.copy()
            ps["attr"] = attr
            ps["model"] = model_name
            all_pair_summ.append(ps)

        st = species_table.copy()
        st["attr"] = attr
        st["model"] = model_name
        all_species.append(st)

        print(model_name, attr, "test_acc=", round(info["test_acc"], 4), "mean_gap=", round(info["mean_gap"], 4))

    info_df = pd.DataFrame(all_info)
    pairs_df = pd.concat(all_pair_summ, ignore_index=True) if all_pair_summ else pd.DataFrame()
    species_df = pd.concat(all_species, ignore_index=True) if all_species else pd.DataFrame()
    return info_df, pairs_df, species_df


In [20]:
# Run the full analysis for both models:
# For each model, run_many returns:
# 1) info_df     : per-attribute summary stats (accuracy, mean gap, etc.)
# 2) pairs_df    : matched-pair recall gap results (controlled evaluation)
# 3) species_df  : per-species prevalence + recall table (overall evaluation)

baseline_info, baseline_pairs, baseline_species = run_many(
    ATTR_LIST, "baseline", BASE_FEAT,
    base_kind_tr, base_ids_tr,
    base_kind_te, base_ids_te,
    layer=LAYER,
    thr=0.5,
    n_pairs=200,
    B_gap=100,
    B_species=100,
)

cbm_info, cbm_pairs, cbm_species = run_many(
    ATTR_LIST, "cbm", CBM_FEAT,
    cbm_kind_tr, cbm_ids_tr,
    cbm_kind_te, cbm_ids_te,
    layer=LAYER,
    thr=0.5,
    n_pairs=200,
    B_gap=100,
    B_species=100,
)

# Quick sanity check: compare attribute-level summaries
baseline_info, cbm_info


baseline has_upper_tail_color::white test_acc= 0.8631 mean_gap= 0.3758
baseline has_bill_shape::all-purpose test_acc= 0.6593 mean_gap= 0.3343
baseline has_breast_pattern::solid test_acc= 0.6748 mean_gap= 0.2879
baseline has_upperparts_color::black test_acc= 0.6921 mean_gap= 0.3016
baseline has_bill_length::shorter_than_head test_acc= 0.7316 mean_gap= 0.3111
baseline has_upperparts_color::brown test_acc= 0.7468 mean_gap= 0.2153
baseline has_wing_color::black test_acc= 0.632 mean_gap= 0.2372
baseline has_breast_color::white test_acc= 0.7401 mean_gap= 0.2152
baseline has_wing_color::grey test_acc= 0.6729 mean_gap= 0.3036
baseline has_wing_color::white test_acc= 0.7527 mean_gap= 0.276
baseline has_under_tail_color::black test_acc= 0.593 mean_gap= 0.1832
baseline has_under_tail_color::white test_acc= 0.7333 mean_gap= 0.2054
cbm has_upper_tail_color::white test_acc= 0.7839 mean_gap= 0.2453
cbm has_bill_shape::all-purpose test_acc= 0.694 mean_gap= 0.3122
cbm has_breast_pattern::solid test_acc

(                                  attr     layer  n_train  n_test  \
 0          has_upper_tail_color::white  layer4.0     5994    5794   
 1          has_bill_shape::all-purpose  layer4.0     5994    5794   
 2            has_breast_pattern::solid  layer4.0     5994    5794   
 3          has_upperparts_color::black  layer4.0     5994    5794   
 4   has_bill_length::shorter_than_head  layer4.0     5994    5794   
 5          has_upperparts_color::brown  layer4.0     5994    5794   
 6                has_wing_color::black  layer4.0     5994    5794   
 7              has_breast_color::white  layer4.0     5994    5794   
 8                 has_wing_color::grey  layer4.0     5994    5794   
 9                has_wing_color::white  layer4.0     5994    5794   
 10         has_under_tail_color::black  layer4.0     5994    5794   
 11         has_under_tail_color::white  layer4.0     5994    5794   
 
     train_pos_rate  test_pos_rate  test_acc  thr  epochs  n_pairs  B_gap  \
 0         

In [21]:
baseline_species.to_csv("baseline_species.csv", index=False)
cbm_species.to_csv("cbm_species.csv", index=False)
baseline_species.sort_values("tp", ascending=False).head(30)
cbm_species.sort_values("tp", ascending=False).head(30)

,species_id,species_name,n,n_pos,n_pred_pos,n_neg,prevalence,tp,recall,precision,recall_bs_mean,recall_ci_lo,recall_ci_hi,recall_ci_width,B,attr,model
717,29,American Crow,30,30,30,0,1.000000,30,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,100,has_upperparts_color::black,cbm
448,177,Prothonotary Warbler,30,30,30,0,1.000000,30,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,100,has_breast_pattern::solid,cbm
1470,146,Forsters Tern,30,29,30,1,0.966667,29,1.000000,0.966667,1.000000,1.000000,1.000000,0.000000,100,has_breast_color::white,cbm
1317,29,American Crow,30,29,30,1,0.966667,29,1.000000,0.966667,1.000000,1.000000,1.000000,0.000000,100,has_wing_color::black,cbm
1856,189,Red bellied Woodpecker,30,30,29,0,1.000000,29,0.966667,1.000000,0.967667,0.915833,1.000000,0.084167,100,has_wing_color::white,cbm
658,192,Downy Woodpecker,30,29,29,1,0.966667,29,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,100,has_upperparts_color::black,cbm
1878,159,Black and white Warbler,30,28,30,2,0.933333,28,1.000000,0.933333,1.000000,1.000000,1.000000,0.000000,100,has_wing_color::white,cbm
1256,189,Red bellied Woodpecker,30,29,29,1,0.966667,28,0.965517,0.965517,0.958640,0.859483,1.000000,0.140517,100,has_wing_color::black,cbm
918,35,Purple Finch,30,28,30,2,0.933333,28,1.000000,0.933333,1.000000,1.000000,1.000000,0.000000,100,has_bill_length::shorter_than_head,cbm
454,186,Cedar Waxwing,30,28,30,2,0.933333,28,1.000000,0.933333,1.000000,1.000000,1.000000,0.000000,100,has_breast_pattern::solid,cbm


Pairs from below

In [22]:
# Collapses the per-(attr, species pair) table into an attribute-level summary:
# For each (model, attr), computes:
# - gap_mean: average gap_mean across all species pairs for that attribute
# - gap_max: the maximum gap_mean pair (worst disparity) for that attribute
# - n_pairs: number of evaluated species pairs for that attribute
#
# Then concatenates baseline + cbm summaries into one table for easy comparison.

def summarize_by_attr(pairs_df: pd.DataFrame):
    """
    Collapses per-(attr, species pair) into per-attribute summary.
    Uses the pair-level bootstrap outputs:
      - gap_mean (per pair)
      - gap_ci_lo/hi (per pair)
      - gap_p (per pair)
      - gap_snr (per pair)

    Output:
      - gap_mean: avg gap_mean over pairs
      - gap_median: median gap_mean over pairs
      - gap_max: worst pair gap_mean
      - frac_p_small: fraction of pairs with p <= 0.05
      - frac_ci_above0: fraction of pairs with CI lower bound > 0
      - gap_snr_mean: avg stability across pairs
    """
    if pairs_df.empty:
        return pairs_df

    g = pairs_df.groupby(["model", "attr"], as_index=False)

    out = g.agg(
        gap_mean=("gap_mean","mean"),
        gap_median=("gap_mean","median"),
        gap_max=("gap_mean","max"),
        n_pairs=("gap_mean","size"),
        frac_p_small=("gap_p", lambda s: float(np.mean(np.asarray(s) <= 0.05))),
        frac_ci_above0=("gap_ci_lo", lambda s: float(np.mean(np.asarray(s) > 0))),
        gap_snr_mean=("gap_snr","mean"),
    ).sort_values(["model", "gap_mean"], ascending=[True, False])

    return out

def top_pairs(pairs_df: pd.DataFrame, model: str, attr: str, k=10):
    """
    Returns top-k pairs by gap_mean for a given (model, attr).

    Columns:
      - gap_mean: average abs(recall_A - recall_B) over B bootstrap resamples
      - gap_ci_lo/hi: 95% bootstrap CI for the gap
      - gap_p: bootstrap p-value for H0: gap==0 (two-sided)
      - gap_std: std of gap over bootstrap resamples
      - gap_snr: gap_mean / gap_std (higher = more stable)
      - gap_norm: same as gap_mean (gap already in [0,1])
      - npos/nneg: matched positives/negatives per species in evaluation slice
      - n_runs: number of bootstrap runs (B)
    """
    sub = pairs_df[(pairs_df["model"] == model) & (pairs_df["attr"] == attr)].copy()
    if sub.empty:
        return sub

    cols = [
        "species_A", "species_B",
        "gap_mean", "gap_ci_lo", "gap_ci_hi", "gap_p",
        "gap_std", "gap_snr", "gap_norm",
        "npos", "nneg", "n_runs",
    ]
    cols = [c for c in cols if c in sub.columns]
    return sub.sort_values("gap_mean", ascending=False).head(k)[cols]



summary = pd.concat([
    summarize_by_attr(baseline_pairs),
    summarize_by_attr(cbm_pairs)
], ignore_index=True)

summary


,model,attr,gap_mean,gap_median,gap_max,n_pairs,frac_p_small,frac_ci_above0,gap_snr_mean
0,baseline,has_upper_tail_color::white,0.375770,0.369167,0.823000,173,0.768786,0.768786,4.220394e+10
1,baseline,has_bill_shape::all-purpose,0.334323,0.295455,1.000000,198,0.828283,0.828283,6.431404e+10
2,baseline,has_bill_length::shorter_than_head,0.311107,0.250000,1.000000,195,0.784615,0.794872,7.198453e+10
3,baseline,has_wing_color::grey,0.303559,0.250786,0.900000,198,0.777778,0.777778,5.466625e+10
4,baseline,has_upperparts_color::black,0.301571,0.267417,0.882000,198,0.772727,0.777778,3.161834e+10
5,baseline,has_breast_pattern::solid,0.287898,0.212086,0.866667,200,0.750000,0.750000,3.884592e+10
6,baseline,has_wing_color::white,0.275964,0.217468,0.800000,194,0.706186,0.711340,5.353928e+10
7,baseline,has_wing_color::black,0.237226,0.208462,0.856000,199,0.708543,0.718593,7.648450e+10
8,baseline,has_upperparts_color::brown,0.215279,0.148132,0.727273,196,0.704082,0.714286,6.302510e+10
9,baseline,has_breast_color::white,0.215195,0.194167,0.805000,197,0.614213,0.614213,7.123219e+09


In [23]:
def add_gap_interpretability_cols(pair_df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds interpretability columns to a *pair_summary* dataframe:
      - gap_snr  = gap_mean / gap_std  (higher = more stable across bootstrap runs)
      - gap_norm = gap_mean / max_possible_gap

    max_possible_gap explanation (why this is a reasonable scale):
      For a fixed threshold thr, prevalence p = P(y=1) puts a hard upper bound on recall differences.
      If the classifier predicts positive on fraction r = P(pred=1), then:
        recall = P(pred=1 | y=1) <= min(1, r/p)
      So across two species with prevalences pA, pB (estimated in matched subset as mpos/(mpos+mneg)),
      the absolute recall gap is bounded by:
        max_gap <= |min(1, r/pA) - min(1, r/pB)|
      We approximate r by the threshold under a well-calibrated model as ~thr (rough heuristic),
      but we can instead just use a safe upper bound:
        max_gap <= 1.0
      To avoid pretending we know r exactly, we use a simple *prevalence-only* bound:
        max_gap_prevalence = 1.0  (conservative)
      and keep gap_norm mainly as “gap_mean on a 0..1 scale”.

    If you later want a tighter bound, pass in the actual per-species predicted-positive rate.
    """
    out = pair_df.copy()

    # Stability: mean gap relative to its bootstrap variability.
    out["gap_snr"] = out["gap_mean"] / out["gap_std"].replace(0, np.nan)

    # Matched prevalence within each species subset is the same by construction:
    # prevalence_matched = mpos / (mpos + mneg)
    # This isn't the *dataset* prevalence, but it's the prevalence of the evaluation slice.
    prev_matched = out["npos"] / (out["npos"] + out["nneg"])
    out["prev_matched"] = prev_matched.astype(float)

    # Conservative normalization (0..1 scale). This avoids overclaiming a "true" max gap.
    out["gap_norm"] = out["gap_mean"] / 1.0

    return out

### Interpreting matched-pair gap columns

Each row corresponds to a *species pair* evaluated for a single attribute and model.

- **gap_mean**  
  Mean absolute difference in recall between the two species, averaged over repeated
  matched-pair resampling runs.  
  *This is the raw recall gap.*

- **gap_std**  
  Standard deviation of the recall gap across repeated runs with different random seeds.  
  *Measures how stable the gap estimate is.*

- **gap_norm**  
  `gap_mean` normalized by the attribute’s typical cross-species recall spread
  (defined as the 90th–10th percentile recall difference across species).  
  *(Is this pair’s gap large relative to how much this attribute usually varies
  across species?)*  
  Values near 1 indicate an extreme pair; values near 0 indicate negligible disparity.

- **gap_snr**  
  Signal-to-noise ratio of the gap: `gap_mean / gap_std`.  
  *Answers: “Is the gap consistently observed, or within noise?”*  
  Larger values indicate a stable, repeatable gap.

- **npos / nneg**  
  Number of positive and negative examples per species used in each matched subset.  
  *Ensures both species are compared under equal prevalence.*

- **n_runs**  
  Number of matched-pair resampling runs used to estimate the gap.  
  *Higher values increase confidence in `gap_mean` and `gap_std`.*

Overall, **gap_norm** indicates *magnitude* (how large the disparity is),
while **gap_snr** indicates *reliability* (how confident we are it is not noise).


In [24]:
# Utility to display the "worst" (largest gap_mean) species pairs for a given attribute and model:
# Filters to (model, attr), sorts by gap_mean descending, prints the top-k pairs.

def top_pairs(pairs_df: pd.DataFrame, model: str, attr: str, k=10):
    """
    Filters to (model, attr), sorts by gap_mean descending, returns top-k pairs.

    Interpreting new columns:
      - gap_ci_lo / gap_ci_hi: bootstrap CI over matched resamples (seeds). If CI excludes 0 => stable gap.
      - gap_p: bootstrap p-value for H0: gap==0 (two-sided).
      - gap_snr: mean gap / std gap (higher => more stable across resamples).
      - gap_norm: gap_mean on 0..1 scale (currently conservative; 0.2 = 20 percentage-point recall gap).
    """
    sub = pairs_df[(pairs_df["model"] == model) & (pairs_df["attr"] == attr)].copy()
    if sub.empty:
        return sub

    cols = [
      "species_A","species_B",
      "gap_mean","gap_ci_lo","gap_ci_hi","gap_p",
      "gap_std","gap_snr","gap_norm","gap_u",
      "npos","nneg","n_runs"
    ]

    # keep only columns that exist (safe if you run old cached tables)
    cols = [c for c in cols if c in sub.columns]

    return sub.sort_values("gap_mean", ascending=False).head(k)[["species_A","species_B","gap_mean","gap_ci_lo","gap_ci_hi","gap_p", 
                                                                 "gap_std","gap_snr","gap_norm","npos","nneg","n_runs"]]



for a in ATTR_LIST:
    print("\nAttribute:", a)
    print("Baseline top pairs:")
    display(top_pairs(baseline_pairs, "baseline", a, k=10))
    print("CBM top pairs:")
    display(top_pairs(cbm_pairs, "cbm", a, k=10))



Attribute: has_upper_tail_color::white
Baseline top pairs:


,species_A,species_B,gap_mean,gap_ci_lo,gap_ci_hi,gap_p,gap_std,gap_snr,gap_norm,npos,nneg,n_runs
0,Clark Nutcracker,Elegant Tern,0.823000,0.80,0.900000,0.0,0.042295,19.458446,0.823000,10,18,100
3,Blue Jay,Red legged Kittiwake,0.808000,0.80,0.900000,0.0,0.027266,29.633985,0.808000,10,12,100
1,Red legged Kittiwake,Clark Nutcracker,0.808000,0.80,0.900000,0.0,0.027197,29.708724,0.808000,10,12,200
2,Red legged Kittiwake,Blue Jay,0.808000,0.80,0.900000,0.0,0.027266,29.633985,0.808000,10,12,100
4,Western Gull,American Goldfinch,0.787500,0.75,0.916667,0.0,0.047930,16.430234,0.787500,12,15,100
5,American Goldfinch,Artic Tern,0.784167,0.75,0.916667,0.0,0.047518,16.502386,0.784167,12,14,100
6,Artic Tern,Blue Jay,0.769000,0.70,0.900000,0.0,0.069187,11.114773,0.769000,10,14,100
7,Blue Jay,Western Gull,0.768000,0.70,0.900000,0.0,0.066485,11.551547,0.768000,10,15,100
8,Clark Nutcracker,Western Gull,0.768000,0.70,0.900000,0.0,0.066317,11.580681,0.768000,10,15,200
9,Elegant Tern,Downy Woodpecker,0.760833,0.75,0.833333,0.0,0.028166,27.012021,0.760833,12,17,100


CBM top pairs:


,species_A,species_B,gap_mean,gap_ci_lo,gap_ci_hi,gap_p,gap_std,gap_snr,gap_norm,npos,nneg,n_runs
0,Red legged Kittiwake,Blue Jay,0.700,0.7,0.7,0.0,0.000000,7.000000e+11,0.700,10,12,100
2,Blue Jay,Red legged Kittiwake,0.700,0.7,0.7,0.0,0.000000,7.000000e+11,0.700,10,12,100
3,Blue Jay,Western Gull,0.700,0.7,0.7,0.0,0.000000,7.000000e+11,0.700,10,15,100
1,Black and white Warbler,Blue Jay,0.700,0.7,0.7,0.0,0.000000,7.000000e+11,0.700,10,16,100
4,Pied Kingfisher,Blue Jay,0.660,0.6,0.7,0.0,0.049237,1.340466e+01,0.660,10,11,100
5,Blue Jay,Pied Kingfisher,0.660,0.6,0.7,0.0,0.049113,1.343847e+01,0.660,10,11,200
6,Blue Jay,Forsters Tern,0.640,0.6,0.7,0.0,0.049237,1.299846e+01,0.640,10,14,100
7,Caspian Tern,Blue Jay,0.629,0.6,0.7,0.0,0.045605,1.379241e+01,0.629,10,17,100
8,Blue Jay,Ring billed Gull,0.614,0.6,0.7,0.0,0.034874,1.760649e+01,0.614,10,18,100
12,Black Tern,Red legged Kittiwake,0.600,0.6,0.6,0.0,0.000000,6.000000e+11,0.600,10,12,100



Attribute: has_bill_shape::all-purpose
Baseline top pairs:


...

,species_A,species_B,gap_mean,gap_ci_lo,gap_ci_hi,gap_p,gap_std,gap_snr,gap_norm,npos,nneg,n_runs
1948,Cape Glossy Starling,Bewick Wren,0.857143,0.857143,0.857143,0.0,0.000000,8.571429e+11,0.857143,14,16,100
1949,Bewick Wren,American Redstart,0.752143,0.714286,0.857143,0.0,0.042416,1.773254e+01,0.752143,14,11,100
1950,Bewick Wren,Orchard Oriole,0.750714,0.714286,0.857143,0.0,0.042465,1.767862e+01,0.750714,14,10,100
1951,Bewick Wren,Magnolia Warbler,0.741429,0.714286,0.785714,0.0,0.037687,1.967322e+01,0.741429,14,11,100
1952,Bewick Wren,Black and white Warbler,0.732857,0.714286,0.785714,0.0,0.033085,2.215071e+01,0.732857,14,14,100
1953,Cape May Warbler,Baltimore Oriole,0.700000,0.700000,0.700000,0.0,0.000000,7.000000e+11,0.700000,10,18,100
1954,Cedar Waxwing,Clark Nutcracker,0.700000,0.700000,0.700000,0.0,0.000000,7.000000e+11,0.700000,10,11,100
1955,Pied Kingfisher,Western Wood Pewee,0.674545,0.636364,0.727273,0.0,0.045095,1.495832e+01,0.674545,11,12,100
1956,Pied Kingfisher,Cape May Warbler,0.647000,0.600000,0.700000,0.0,0.050161,1.289838e+01,0.647000,10,12,100
1957,Red bellied Woodpecker,Cedar Waxwing,0.645000,0.600000,0.700000,0.0,0.050000,1.290000e+01,0.645000,10,13,100



Attribute: has_under_tail_color::white
Baseline top pairs:


,species_A,species_B,gap_mean,gap_ci_lo,gap_ci_hi,gap_p,gap_std,gap_snr,gap_norm,npos,nneg,n_runs
2146,Elegant Tern,Cactus Wren,0.600000,0.6,0.600000,0.0,0.000000,6.000000e+11,0.600000,10,13,100
2148,Cactus Wren,Artic Tern,0.550000,0.5,0.600000,0.0,0.050252,1.094486e+01,0.550000,10,10,100
2147,Artic Tern,Cactus Wren,0.550000,0.5,0.600000,0.0,0.050252,1.094486e+01,0.550000,10,10,100
2149,Cactus Wren,Western Gull,0.549000,0.5,0.600000,0.0,0.050242,1.092715e+01,0.549000,10,12,100
2150,Cactus Wren,Forsters Tern,0.546000,0.5,0.600000,0.0,0.050091,1.090020e+01,0.546000,10,11,100
2151,Cactus Wren,Black and white Warbler,0.544000,0.5,0.600000,0.0,0.049889,1.090426e+01,0.544000,10,11,100
2152,Clark Nutcracker,Ring billed Gull,0.532500,0.5,0.583333,0.0,0.040748,1.306818e+01,0.532500,12,10,200
2153,Clark Nutcracker,Herring Gull,0.531667,0.5,0.583333,0.0,0.040653,1.307828e+01,0.531667,12,10,100
2154,Clark Nutcracker,Western Gull,0.530833,0.5,0.583333,0.0,0.040436,1.312762e+01,0.530833,12,12,100
2155,Western Gull,Clark Nutcracker,0.530833,0.5,0.583333,0.0,0.040335,1.316073e+01,0.530833,12,12,200


CBM top pairs:


,species_A,species_B,gap_mean,gap_ci_lo,gap_ci_hi,gap_p,gap_std,gap_snr,gap_norm,npos,nneg,n_runs
2146,Red legged Kittiwake,Chipping Sparrow,0.333333,0.333333,0.333333,0.0,0.0,3.333333e+11,0.333333,12,11,100
2148,Ring billed Gull,Chipping Sparrow,0.333333,0.333333,0.333333,0.0,0.0,3.333333e+11,0.333333,12,10,100
2149,Chipping Sparrow,Ring billed Gull,0.333333,0.333333,0.333333,0.0,0.0,3.333333e+11,0.333333,12,10,100
2150,Chipping Sparrow,Red legged Kittiwake,0.333333,0.333333,0.333333,0.0,0.0,3.333333e+11,0.333333,12,11,100
2147,Pied Kingfisher,Chipping Sparrow,0.333333,0.333333,0.333333,0.0,0.0,3.333333e+11,0.333333,12,17,100
2154,Cactus Wren,Red cockaded Woodpecker,0.300000,0.300000,0.300000,0.0,0.0,3.000000e+11,0.300000,10,17,100
2155,Cactus Wren,Black and white Warbler,0.300000,0.300000,0.300000,0.0,0.0,3.000000e+11,0.300000,10,11,100
2156,Cactus Wren,Artic Tern,0.300000,0.300000,0.300000,0.0,0.0,3.000000e+11,0.300000,10,10,100
2153,Cactus Wren,Western Gull,0.300000,0.300000,0.300000,0.0,0.0,3.000000e+11,0.300000,10,12,100
2152,Artic Tern,Cactus Wren,0.300000,0.300000,0.300000,0.0,0.0,3.000000e+11,0.300000,10,10,100


### How confidence intervals and p-values are computed

For each species pair and attribute, we run the matched-pair evaluation multiple times
(`n_runs`, using different random seeds).  
Each run produces one recall gap value.  
These gap values form an *empirical sampling distribution* of the recall gap.

#### Bootstrap confidence interval (CI)

- We treat the list of gap values from the repeated matched resampling runs as a
  bootstrap distribution.
- The **95% confidence interval** is computed using the *percentile method*:
  - `gap_ci_lo` = 2.5th percentile of the gap values
  - `gap_ci_hi` = 97.5th percentile of the gap values
- Interpretation:
  - If the interval **excludes 0**, the recall gap is stable under resampling.
  - Narrow intervals indicate low sampling variability; wide intervals indicate
    uncertainty due to limited data or few runs.

#### Bootstrap p-value

- We test the null hypothesis **H₀: recall gap = 0**.
- The p-value is computed directly from the empirical gap distribution:
  - Compute the fraction of runs where the gap is ≤ 0
  - Compute the fraction of runs where the gap is ≥ 0
  - The two-sided p-value is  
    `p = 2 × min(P(gap ≤ 0), P(gap ≥ 0))`
- This p-value measures how often the resampled gaps are consistent with no difference
  in recall between the two species.

- A gap is considered **statistically significant** if:
  - `gap_p ≤ 0.05`, **and**
  - the confidence interval `[gap_ci_lo, gap_ci_hi]` does not include `0`
- In practice, we also check effect size:
  - very small gaps can be statistically significant with enough resamples, but are
    not substantively meaningful
  - therefore, significance is interpreted jointly with `gap_mean` and `gap_snr`

**Important note:**  
The bootstrap distribution here reflects variability induced by *matched resampling*,
not independent image-level noise. This avoids parametric assumptions that do not hold
in fine-grained species datasets.


What to do next


### 1. Compare layers relative to species emergence

Run the same matched-pair test at:
- layer3.x
- layer4.0
- avgpool

If recall gaps increase after the species-emergence layer, this supports the idea that
species representations are feeding back into attribute prediction?



### 2. Aggregate across attributes

Instead of looking at single attributes:
- Compute mean gap across all attributes
- Measure fraction of pairs with gap > 0.3

This shows whether entanglement is a general phenomenon or limited to color attributes.



### 3. Negative control attributes

Test attributes that should be weakly species-correlated (e.g. rare shapes).

Why:
If gaps shrink for these attributes, it strengthens the causal interpretation
that species identity drives
